# Гибридный рекомендательный движок

Здесь мы строим уже не набор ручных правил, а настоящий рекомендательный движок, который сочетает:

- историю взаимодействий клиента с товарами;
- похожесть товаров;
- популярность товара;
- качество товара через штраф за возвраты.

Это намного сильнее для конкурсного решения, чем просто рекомендовать безопасные категории вручную.

## Как работает движок

Мы используем гибридный подход:

1. Строим матрицу `клиент -> товар` с весами взаимодействий.
2. Сжимаем её в скрытые факторы через `TruncatedSVD`.
3. Считаем похожесть товаров по их скрытым представлениям.
4. Для каждого клиента набираем кандидатов из похожих товаров.
5. Добавляем бонус за популярность и знакомые категории.
6. Штрафуем товары и категории с высоким уровнем возвратов.

То есть это уже настоящий рекомендательный двигатель, а не просто список популярных товаров.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from recommender_engine import build_hybrid_recommender

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data.csv"


In [ ]:
recommender = build_hybrid_recommender(DATA_PATH, min_item_interactions=20, n_components=40)
len(recommender.product_stats), recommender.user_history['user_id'].nunique()


## 1. Что знает движок о каталоге

In [ ]:
summary = pd.Series(
    {
        "товаров_в_движке": len(recommender.product_stats),
        "клиентов_с_историей": recommender.user_history['user_id'].nunique(),
        "средний_возврат_по_товарам": round(recommender.product_stats['return_rate'].mean(), 4),
    }
).to_frame("value")
summary


In [ ]:
recommender.product_stats.sort_values('safe_popularity_score', ascending=False).head(15)


## 2. Примеры рекомендаций для клиентов

In [ ]:
sample_users = recommender.user_history['user_id'].drop_duplicates().head(5).tolist()
sample_users


In [ ]:
for user_id in sample_users:
    print(f'\n=== user_id = {user_id} ===')
    display(recommender.recommend_for_user(user_id=user_id, top_k=5)[['product_name_clean', 'category', 'brand', 'avg_price', 'return_rate', 'hybrid_score', 'reason']])


## 3. Режим для клиентов с высоким риском ухода

Для таких клиентов мы делаем рекомендации осторожнее:

- усиливаем знакомые категории;
- сильнее штрафуем товары с возвратами;
- поднимаем более надёжные товары.

In [ ]:
for user_id in sample_users:
    print(f'\n=== high-risk user_id = {user_id} ===')
    display(recommender.recommend_for_user(user_id=user_id, top_k=5, high_risk=True)[['product_name_clean', 'category', 'brand', 'avg_price', 'return_rate', 'hybrid_score', 'reason']])


## Почему это сильнее для проекта

Теперь в проекте есть полноценный рекомендательный сервис, а не только логика на правилах.

Это важно для защиты, потому что можно честно говорить:

- мы построили модель ухода;
- мы построили рекомендательный движок;
- мы связали их между собой;
- для клиентов с высоким риском ухода рекомендации переранжируются по безопасным и подходящим товарам.

Именно такая связка выглядит как продвинутое решение, а не как просто EDA с парой моделей.

## Что дальше

Следующий шаг — встроить этот движок в дашборд вместо текущей упрощённой логики рекомендаций. Тогда в демонстрации будет настоящий recsys-блок.